<a href="https://colab.research.google.com/github/jahnavi13383/javascript-coding/blob/main/CropDiseaseMappingSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌿 Crop Disease Report Mapping System
### Complete Pipeline — MobileNetV2 + PlantVillage + Interactive Dashboard
**Methodology:** Data Collection → Preprocessing → Disease Detection (CNN) → Report Generation → Mapping → Outbreak Analysis → Alerts → Dashboard → Feedback

---

## 📦 STEP 0 — Install Dependencies

In [1]:
!pip install folium scikit-learn seaborn plotly ipywidgets Pillow tqdm scipy scikit-image -q
print('✅ All packages installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.9 MB/s eta 0:00:00
✅ All packages installed.


## 🔗 STEP 1 — Mount Google Drive & Dataset Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted.')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os, glob

# 🔧 UPDATE THIS PATH if your PlantVillage folder is elsewhere
DATASET_ROOT = '/content/drive/MyDrive/PlantVillage'

# Auto-discover nested class folders
class_folders = []
for root, dirs, files in os.walk(DATASET_ROOT):
    if any(f.lower().endswith(('.jpg','.jpeg','.png')) for f in files):
        class_folders.append(root)

print(f'📁 Dataset root  : {DATASET_ROOT}')
print(f'🌿 Classes found : {len(class_folders)}')
for f in sorted(class_folders):
    n = len([x for x in os.listdir(f) if x.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f'   {os.path.basename(f):45s}  →  {n:5d} images')

## ⚙️ STEP 2 — Imports & Global Configuration

In [ ]:
import os, json, random, warnings, datetime, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.cluster import DBSCAN

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2

import folium
from folium.plugins import HeatMap, MarkerCluster
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
np.random.seed(42); random.seed(42)

# ── Paths
OUTPUT_DIR  = '/content/crop_disease_system'
MODEL_PATH  = f'{OUTPUT_DIR}/mobilenetv2_plantvillage.h5'
REPORTS_CSV = f'{OUTPUT_DIR}/disease_reports.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Hyper-params
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 15
LR         = 1e-4

# ── Severity colour palette
SEVERITY_COLORS = {
    'Low':'#00cc44', 'Moderate':'#ffcc00',
    'High':'#ff8800', 'Severe':'#cc0000'
}

print('✅ Imports done | TF:', tf.__version__)
print('🖥  GPU:', tf.config.list_physical_devices('GPU'))

## 📊 STEP 3 — Exploratory Data Analysis

In [ ]:
# Build class → images index (handles nested PlantVillage structure)
class_image_map = {}
for folder in sorted(class_folders):
    cls  = os.path.basename(folder)
    imgs = [os.path.join(folder, f) for f in os.listdir(folder)
            if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if imgs:
        class_image_map[cls] = imgs

CLASS_NAMES  = sorted(class_image_map.keys())
NUM_CLASSES  = len(CLASS_NAMES)
class_to_idx = {c:i for i,c in enumerate(CLASS_NAMES)}
idx_to_class = {i:c for c,i in class_to_idx.items()}
counts       = {c: len(v) for c,v in class_image_map.items()}

print(f'Total classes : {NUM_CLASSES}')
print(f'Total images  : {sum(counts.values()):,}')

# ── Class distribution plot
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
sc = dict(sorted(counts.items(), key=lambda x: -x[1]))
axes[0].barh(list(sc.keys()), list(sc.values()), color='#2d7a2d')
axes[0].set_xlabel('Number of Images')
axes[0].set_title('PlantVillage — Images per Class', fontweight='bold')
axes[0].tick_params(axis='y', labelsize=7)
axes[1].pie(list(counts.values()), autopct='%1.1f%%',
            startangle=90, colors=plt.cm.tab20.colors)
axes[1].set_title('Class Distribution (%)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample images grid
sample_cls = CLASS_NAMES[:12]
fig, axes  = plt.subplots(3, 4, figsize=(16, 12))
for ax, cls in zip(axes.flatten(), sample_cls):
    img = Image.open(random.choice(class_image_map[cls])).resize((224,224))
    ax.imshow(img)
    ax.set_title(cls.replace('_',' '), fontsize=7, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample Images — PlantVillage', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔄 STEP 4 — Data Preprocessing (Methodology Step 2)

In [ ]:
# ── Image quality validation
def validate_image(path, min_size=50):
    try:
        img = Image.open(path)
        w, h = img.size
        return w >= min_size and h >= min_size
    except:
        return False

# ── Build full file list
all_paths, all_labels = [], []
for cls, paths in class_image_map.items():
    for p in paths:
        if validate_image(p):
            all_paths.append(p)
            all_labels.append(class_to_idx[cls])

print(f'✅ Valid images: {len(all_paths):,}')

# ── 70 / 15 / 15 split
X_train, X_test, y_train, y_test = train_test_split(
    all_paths, all_labels, test_size=0.15, random_state=42, stratify=all_labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.176, random_state=42, stratify=y_train)

print(f'Train: {len(X_train):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}')

In [ ]:
# ── tf.data pipeline with augmentation
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_dataset(paths, labels, augment_data=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths), tf.constant(labels)))
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train, y_train, augment_data=True)
val_ds   = make_dataset(X_val,   y_val)
test_ds  = make_dataset(X_test,  y_test)
print('✅ tf.data pipelines ready.')

## 🧠 STEP 5 — MobileNetV2 Disease Detection Model (Methodology Step 3)

In [ ]:
def build_mobilenetv2(num_classes):
    base = MobileNetV2(input_shape=(IMG_SIZE,IMG_SIZE,3),
                       include_top=False, weights='imagenet')
    base.trainable = False
    inp = layers.Input(shape=(IMG_SIZE,IMG_SIZE,3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inp, out)
    model.compile(optimizer=optimizers.Adam(LR),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model, base

model, base_model = build_mobilenetv2(NUM_CLASSES)
model.summary()

In [ ]:
cb_list = [
    callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True,
                              monitor='val_accuracy', verbose=1),
    callbacks.EarlyStopping(monitor='val_loss', patience=5,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=3, min_lr=1e-7, verbose=1),
]

# ── Phase 1: frozen base
print('=' * 55)
print('Phase 1 — Training top layers (base frozen)')
print('=' * 55)
history1 = model.fit(train_ds, validation_data=val_ds,
                     epochs=10, callbacks=cb_list, verbose=1)

In [ ]:
# ── Phase 2: fine-tune last 50 layers
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(optimizer=optimizers.Adam(LR/10),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print('=' * 55)
print('Phase 2 — Fine-tuning last 50 layers')
print('=' * 55)
history2 = model.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS, callbacks=cb_list, verbose=1)

In [ ]:
# Training curves
def plot_history(h1, h2=None):
    acc  = h1.history['accuracy']  + (h2.history['accuracy']  if h2 else [])
    vacc = h1.history['val_accuracy'] + (h2.history['val_accuracy'] if h2 else [])
    loss = h1.history['loss']      + (h2.history['loss']      if h2 else [])
    vloss= h1.history['val_loss']  + (h2.history['val_loss']  if h2 else [])
    ep   = range(1, len(acc)+1)
    split= len(h1.history['accuracy'])

    fig,(ax1,ax2) = plt.subplots(1,2,figsize=(14,5))
    ax1.plot(ep,acc,'#2d7a2d',label='Train'); ax1.plot(ep,vacc,'--#ff8800',label='Val')
    ax1.axvline(split,color='gray',linestyle=':',label='Fine-tune start')
    ax1.set_title('Accuracy',fontweight='bold'); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(ep,loss,'#2d7a2d',label='Train'); ax2.plot(ep,vloss,'--#cc0000',label='Val')
    ax2.axvline(split,color='gray',linestyle=':')
    ax2.set_title('Loss',fontweight='bold'); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/training_history.png',dpi=150,bbox_inches='tight')
    plt.show()

plot_history(history1, history2)

## 📋 STEP 6 — Model Evaluation

In [ ]:
best_model = tf.keras.models.load_model(MODEL_PATH)

loss, acc = best_model.evaluate(test_ds, verbose=0)
print(f'\n🎯 Test Accuracy : {acc*100:.2f}%')
print(f'📉 Test Loss     : {loss:.4f}')

y_pred_prob = best_model.predict(test_ds, verbose=1)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = np.concatenate([y for _,y in test_ds], axis=0)

print('\n📊 Classification Report:')
print(classification_report(y_true, y_pred,
      target_names=[c[:30] for c in CLASS_NAMES]))

In [ ]:
# Confusion matrix (top 15 classes)
top_n = min(15, NUM_CLASSES)
top_idx = np.argsort(np.bincount(y_true))[-top_n:]
mask    = np.isin(y_true, top_idx)
cm      = confusion_matrix(y_true[mask], y_pred[mask], labels=top_idx)

fig, ax = plt.subplots(figsize=(14,12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=[CLASS_NAMES[i][:20] for i in top_idx],
            yticklabels=[CLASS_NAMES[i][:20] for i in top_idx], ax=ax)
ax.set_xlabel('Predicted',fontweight='bold')
ax.set_ylabel('Actual',fontweight='bold')
ax.set_title(f'Confusion Matrix — Top {top_n} Classes',fontweight='bold')
plt.xticks(rotation=45,ha='right',fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png',dpi=150,bbox_inches='tight')
plt.show()

## 🗺️ STEP 7 — Prediction + Report Generator (Methodology Steps 3 & 4)

In [ ]:
def get_severity(conf):
    if conf >= 0.90: return 'Severe'
    elif conf >= 0.75: return 'High'
    elif conf >= 0.55: return 'Moderate'
    else: return 'Low'

def predict_disease(img_path, mdl=best_model):
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE,IMG_SIZE))
    arr = np.expand_dims(np.array(img,dtype=np.float32)/255., 0)
    probs   = mdl.predict(arr, verbose=0)[0]
    top3idx = np.argsort(probs)[-3:][::-1]
    top3    = [(CLASS_NAMES[i], float(probs[i])) for i in top3idx]
    pred    = CLASS_NAMES[top3idx[0]]
    conf    = float(probs[top3idx[0]])
    parts   = pred.split('___')
    crop    = parts[0] if len(parts)>0 else 'Unknown'
    disease = parts[1].replace('_',' ') if len(parts)>1 else pred
    return {'class':pred,'crop':crop,'disease':disease,
            'confidence':conf,'severity':get_severity(conf),'top3':top3}

# ── CSV report database
REPORT_COLS = ['report_id','timestamp','image_path','crop','disease',
               'confidence','severity','latitude','longitude','farmer_name','notes']
if not os.path.exists(REPORTS_CSV):
    pd.DataFrame(columns=REPORT_COLS).to_csv(REPORTS_CSV, index=False)

def save_report(img_path, result, lat, lon, farmer='Unknown', notes=''):
    df  = pd.read_csv(REPORTS_CSV)
    rid = f'RPT{len(df)+1:04d}'
    row = {'report_id':rid,
           'timestamp':datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
           'image_path':img_path, 'crop':result['crop'],
           'disease':result['disease'],
           'confidence':round(result['confidence'],4),
           'severity':result['severity'],
           'latitude':lat,'longitude':lon,
           'farmer_name':farmer,'notes':notes}
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(REPORTS_CSV, index=False)
    print(f'✅ Report {rid} saved.')
    return rid

print('✅ Prediction & Report functions ready.')

## 🌍 STEP 8 — Simulate Geo-tagged Reports (Demo Data)

In [ ]:
# Synthetic reports for map demo (replace with real submissions in production)
np.random.seed(0)
BASE_LAT, BASE_LON = 16.5, 80.6
CROP_DISEASES = [
    ('Tomato','Early Blight'),('Tomato','Late Blight'),
    ('Potato','Early Blight'),('Corn','Common Rust'),
    ('Pepper','Bacterial Spot'),('Apple','Apple Scab'),
    ('Grape','Black Rot'),('Tomato','Leaf Mold'),
    ('Cotton','Alternaria'),('Chili','Leaf Curl')
]
demo = []
for i in range(80):
    crop,dis = random.choice(CROP_DISEASES)
    conf = round(random.uniform(0.45,0.99),3)
    lat  = round(BASE_LAT + np.random.normal(0,0.3), 4)
    lon  = round(BASE_LON + np.random.normal(0,0.3), 4)
    hrs  = random.randint(0,168)
    ts   = (datetime.datetime.now() - datetime.timedelta(hours=hrs)).strftime('%Y-%m-%d %H:%M:%S')
    demo.append({'report_id':f'RPT{i+1:04d}','timestamp':ts,'image_path':'',
                 'crop':crop,'disease':dis,'confidence':conf,
                 'severity':get_severity(conf),'latitude':lat,'longitude':lon,
                 'farmer_name':f'Farmer_{i+1}','notes':''})

df_reports = pd.DataFrame(demo)
df_reports.to_csv(REPORTS_CSV, index=False)
print(f'✅ {len(df_reports)} demo reports saved.')
df_reports.head()

## 🗺️ STEP 9 — Mapping & Visualization (Methodology Step 5)

In [ ]:
def build_disease_map(df, filter_severity=None, filter_disease=None,
                      days_filter=None, output_file=None):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    if days_filter:
        cutoff = datetime.datetime.now() - datetime.timedelta(days=days_filter)
        df = df[df['timestamp'] >= cutoff]
    if filter_severity and filter_severity != 'All':
        df = df[df['severity'] == filter_severity]
    if filter_disease and filter_disease != 'All':
        df = df[df['disease'] == filter_disease]

    m = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()],
                   zoom_start=10, tiles='CartoDB positron')

    HeatMap(df[['latitude','longitude']].dropna().values.tolist(),
            radius=20, blur=15, min_opacity=0.4,
            gradient={'0.2':'blue','0.5':'yellow','0.8':'orange','1.0':'red'}).add_to(m)

    cluster = MarkerCluster(name='Disease Reports').add_to(m)
    for _, r in df.iterrows():
        col = SEVERITY_COLORS.get(r['severity'],'#888')
        popup = f"""<div style='font-family:Arial;min-width:180px'>
          <b style='color:{col}'>⚠ {r['severity']} Severity</b><br>
          <b>Crop:</b> {r['crop']}<br><b>Disease:</b> {r['disease']}<br>
          <b>Confidence:</b> {r['confidence']*100:.1f}%<br>
          <b>Farmer:</b> {r['farmer_name']}<br>
          <b>Time:</b> {str(r['timestamp'])[:16]}<br>
          <b>Location:</b> {r['latitude']:.4f}, {r['longitude']:.4f}</div>"""
        folium.CircleMarker(
            location=[r['latitude'],r['longitude']], radius=8,
            color=col, fill=True, fill_opacity=0.85,
            popup=folium.Popup(popup, max_width=250),
            tooltip=f"{r['crop']}: {r['disease']}"
        ).add_to(cluster)

    m.get_root().html.add_child(folium.Element('''
    <div style="position:fixed;bottom:30px;left:30px;z-index:9999;
                background:white;padding:12px;border-radius:8px;
                border:2px solid #2d7a2d;font-family:Arial;font-size:12px">
    <b>Disease Severity</b><br>
    <span style="color:#00cc44">●</span> Low<br>
    <span style="color:#ffcc00">●</span> Moderate<br>
    <span style="color:#ff8800">●</span> High<br>
    <span style="color:#cc0000">●</span> Severe</div>'''))
    folium.LayerControl().add_to(m)
    if output_file:
        m.save(output_file)
        print(f'✅ Map saved: {output_file}')
    return m

disease_map = build_disease_map(df_reports, output_file=f'{OUTPUT_DIR}/disease_map.html')
display(disease_map)

## 📈 STEP 10 — Outbreak Analysis (Methodology Step 6)

In [ ]:
def outbreak_analysis(df):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    fig, axes = plt.subplots(2, 2, figsize=(16,12))

    # Disease frequency
    dc = df.groupby('disease').size().sort_values(ascending=False).head(10)
    axes[0,0].barh(dc.index, dc.values, color='#2d7a2d')
    axes[0,0].set_title('Top 10 Diseases', fontweight='bold')

    # Severity pie
    sc = df['severity'].value_counts()
    axes[0,1].pie(sc.values, labels=sc.index,
                  colors=[SEVERITY_COLORS[s] for s in sc.index],
                  autopct='%1.1f%%', startangle=90)
    axes[0,1].set_title('Severity Distribution', fontweight='bold')

    # Time trend
    df['date'] = df['timestamp'].dt.date
    daily = df.groupby('date').size().reset_index(name='count')
    axes[1,0].plot(daily['date'], daily['count'], marker='o', color='#2d7a2d', lw=2)
    axes[1,0].fill_between(daily['date'], daily['count'], alpha=0.2, color='#2d7a2d')
    axes[1,0].set_title('Daily Report Trend', fontweight='bold')
    axes[1,0].tick_params(axis='x', rotation=30)
    axes[1,0].grid(alpha=0.3)

    # DBSCAN hotspot clustering
    coords = df[['latitude','longitude']].dropna().values
    db = DBSCAN(eps=0.05, min_samples=3).fit(coords)
    df['cluster'] = db.labels_
    n_hot = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    axes[1,1].scatter(df['longitude'], df['latitude'], c=df['cluster'],
                      cmap='tab20', alpha=0.7, s=50, edgecolors='k', lw=0.3)
    axes[1,1].set_title(f'Hotspot Clustering ({n_hot} hotspots)', fontweight='bold')
    axes[1,1].grid(alpha=0.3)

    plt.suptitle('Outbreak Analysis', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/outbreak_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n🔥 Active Hotspots     : {n_hot}')
    print(f'   High/Severe Reports : {len(df[df.severity.isin(["High","Severe"])])}')
    print(f'   Most common disease : {dc.index[0]}')
    return df

df_reports = outbreak_analysis(df_reports)

## 🔔 STEP 11 — Alerts & Notifications (Methodology Step 7)

In [ ]:
def generate_alerts(df):
    alerts = []
    for cid in df['cluster'].unique():
        if cid == -1: continue
        cdf = df[df['cluster']==cid]
        if len(cdf) < 3: continue
        top_dis  = cdf['disease'].value_counts().index[0]
        top_crop = cdf['crop'].value_counts().index[0]
        max_sev  = cdf['severity'].map({'Low':1,'Moderate':2,'High':3,'Severe':4}).max()
        sev_lbl  = {1:'Low',2:'Moderate',3:'High',4:'Severe'}[max_sev]
        lat,lon  = cdf['latitude'].mean(), cdf['longitude'].mean()

        sms = (f"⚠ CROP ALERT: {sev_lbl} risk of {top_dis} in {top_crop} "
               f"near {lat:.3f},{lon:.3f}. ({len(cdf)} reports). Take precautions.")
        email = (f"Subject: Disease Outbreak — {top_dis}\n"
                 f"A {sev_lbl}-severity outbreak of '{top_dis}' affecting {top_crop}\n"
                 f"Location: {lat:.4f}°N, {lon:.4f}°E | Reports: {len(cdf)}\n"
                 f"Recommended: Inspect nearby fields immediately.")
        alerts.append({'cluster_id':cid,'disease':top_dis,'crop':top_crop,
                       'severity':sev_lbl,'reports':len(cdf),
                       'lat':round(lat,4),'lon':round(lon,4),
                       'sms':sms,'email':email})

    icons = {'Low':'🟢','Moderate':'🟡','High':'🟠','Severe':'🔴'}
    print(f'🚨 {len(alerts)} outbreak alerts generated:\n')
    for a in alerts:
        print(f"{icons.get(a['severity'],'⚪')} Cluster {a['cluster_id']} | "
              f"{a['disease']} ({a['crop']}) | {a['severity']} | {a['reports']} reports")
        print(f"   📱 SMS  : {a['sms'][:90]}...")
        print(f"   📧 Mail : {a['email'][:80]}...\n")
    return alerts

alerts = generate_alerts(df_reports)

## 🖥️ STEP 12 — Interactive Dashboard App (Methodology Steps 1, 4, 8)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  COMPLETE INTERACTIVE APP — Upload | Analyse | Map | Alerts
# ═══════════════════════════════════════════════════════════

S  = {'description_width':'130px'}
LW = widgets.Layout(width='400px')
LB = widgets.Layout(width='180px', height='36px')

w_upload = widgets.FileUpload(accept='image/*', multiple=False,
                              description='Upload Image', style=S, layout=LW)
w_farmer = widgets.Text(description='Farmer Name:', style=S, layout=LW,
                        placeholder='Your name')
w_lat    = widgets.FloatText(description='Latitude:',  value=16.506, style=S, layout=LW)
w_lon    = widgets.FloatText(description='Longitude:', value=80.648, style=S, layout=LW)
w_notes  = widgets.Textarea(description='Notes:', style=S, placeholder='Field notes...',
                             layout=widgets.Layout(width='400px', height='60px'))

btn_submit = widgets.Button(description='🔍 Analyse & Submit',
                             button_style='success', layout=LB)
btn_map    = widgets.Button(description='🗺 View Map',
                             button_style='info', layout=LB)
btn_dash   = widgets.Button(description='📊 Dashboard',
                             button_style='warning', layout=LB)
btn_alerts = widgets.Button(description='🔔 Alerts',
                             button_style='danger', layout=LB)

out = widgets.Output()

header = widgets.HTML("""
<div style='background:linear-gradient(135deg,#1a5c1a,#2d9e2d);
            padding:18px;border-radius:10px;color:white;margin-bottom:12px'>
  <h2 style='margin:0'>🌿 Crop Disease Report Mapping System</h2>
  <p style='margin:4px 0 0'>MobileNetV2 | PlantVillage | Real-time Mapping & Alerts</p>
</div>""")

ui = widgets.VBox([
    header,
    widgets.HTML("<b style='color:#1a5c1a'>📸 Step 1: Upload image & enter location</b>"),
    widgets.HBox([widgets.VBox([w_upload, w_farmer, w_lat, w_lon, w_notes]),
                  widgets.VBox([widgets.HTML('<br><br>'), btn_submit])]),
    widgets.HTML('<hr>'),
    widgets.HTML("<b style='color:#1a5c1a'>📊 Step 2: Explore results</b>"),
    widgets.HBox([btn_map, btn_dash, btn_alerts]),
    out
], layout=widgets.Layout(padding='10px'))

# ── Handlers
def on_submit(_):
    with out:
        clear_output(wait=True)
        if not w_upload.value:
            print('⚠️  Please upload a crop image first.')
            return
        info    = list(w_upload.value.values())[0]
        tmp     = f"/tmp/{info['metadata']['name']}"
        with open(tmp,'wb') as f: f.write(info['content'])
        if not validate_image(tmp):
            print('❌ Invalid image.')
            return
        print('🧠 Running MobileNetV2 disease detection...')
        result = predict_disease(tmp)

        fig,(ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
        ax1.imshow(Image.open(tmp).resize((224,224)))
        ax1.set_title('Uploaded Image',fontweight='bold'); ax1.axis('off')
        top3   = result['top3']
        labels = [c[:25] for c,_ in top3]
        scores = [v*100 for _,v in top3]
        ax2.barh(labels[::-1],scores[::-1],color=['#2d7a2d','#88cc44','#ccee88'])
        ax2.set_title('Top-3 Predictions',fontweight='bold')
        ax2.set_xlabel('Confidence (%)')
        for i,(s) in enumerate(scores[::-1]):
            ax2.text(s+0.5,i,f'{s:.1f}%',va='center')
        plt.tight_layout(); plt.show()

        col = SEVERITY_COLORS.get(result['severity'],'#888')
        display(HTML(f"""
        <div style='border:2px solid {col};border-radius:8px;padding:14px;margin:8px 0'>
          <h3 style='color:{col};margin:0'>⚠ {result['severity']} Severity</h3>
          <table><tr><td><b>🌱 Crop</b></td><td>{result['crop']}</td></tr>
          <tr><td><b>🦠 Disease</b></td><td>{result['disease']}</td></tr>
          <tr><td><b>📊 Confidence</b></td><td>{result['confidence']*100:.1f}%</td></tr>
          <tr><td><b>📍 Location</b></td>
          <td>{w_lat.value}°N, {w_lon.value}°E</td></tr></table></div>"""))
        save_report(tmp, result, w_lat.value, w_lon.value,
                    w_farmer.value or 'Anonymous', w_notes.value)

def on_map(_):
    with out:
        clear_output(wait=True)
        df = pd.read_csv(REPORTS_CSV)
        print(f'🗺️  Rendering map for {len(df)} reports...')
        display(build_disease_map(df, output_file=f'{OUTPUT_DIR}/disease_map_live.html'))

def on_dash(_):
    with out:
        clear_output(wait=True)
        df = pd.read_csv(REPORTS_CSV)
        coords = df[['latitude','longitude']].dropna().values
        if len(coords)>=3:
            df['cluster'] = DBSCAN(eps=0.05,min_samples=3).fit(coords).labels_
        else:
            df['cluster'] = -1
        outbreak_analysis(df)

def on_alerts(_):
    with out:
        clear_output(wait=True)
        df = pd.read_csv(REPORTS_CSV)
        coords = df[['latitude','longitude']].dropna().values
        if len(coords)>=3:
            df['cluster'] = DBSCAN(eps=0.05,min_samples=3).fit(coords).labels_
        else:
            df['cluster'] = -1
        generate_alerts(df)

btn_submit.on_click(on_submit)
btn_map.on_click(on_map)
btn_dash.on_click(on_dash)
btn_alerts.on_click(on_alerts)
display(ui)

## 🔁 STEP 13 — Feedback & Model Retraining (Methodology Step 9)

In [ ]:
FEEDBACK_CSV = f'{OUTPUT_DIR}/feedback.csv'
if not os.path.exists(FEEDBACK_CSV):
    pd.DataFrame(columns=['report_id','correct','actual_class','comment']).to_csv(FEEDBACK_CSV,index=False)

def submit_feedback(report_id, correct, actual_class='', comment=''):
    df  = pd.read_csv(FEEDBACK_CSV)
    row = {'report_id':report_id,'correct':correct,
           'actual_class':actual_class,'comment':comment}
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(FEEDBACK_CSV,index=False)
    print(f'✅ Feedback for {report_id} recorded.')

def retraining_check(threshold=0.80):
    df = pd.read_csv(FEEDBACK_CSV)
    if len(df) < 20:
        print('ℹ️  Need ≥20 feedback entries to assess retraining need.')
        return
    acc = df['correct'].astype(bool).mean()
    print(f'📊 User-reported accuracy: {acc*100:.1f}%')
    if acc < threshold:
        print('⚠️  Below threshold — retraining recommended! Re-run STEP 5 with new data.')
    else:
        print('✅ Model performing well.')

# Example usage
submit_feedback('RPT0001', True,  comment='Correctly identified early blight')
submit_feedback('RPT0002', False, 'Late Blight', 'Was late blight, not early')
retraining_check()

## 💾 STEP 14 — Save All Outputs to Drive

In [ ]:
DRIVE_OUT = '/content/drive/MyDrive/CropDiseaseSystemOutputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
for f in os.listdir(OUTPUT_DIR):
    shutil.copy2(os.path.join(OUTPUT_DIR,f), os.path.join(DRIVE_OUT,f))
print('✅ All outputs saved to Google Drive:')
for f in os.listdir(DRIVE_OUT):
    sz = os.path.getsize(os.path.join(DRIVE_OUT,f))
    print(f'   {f:45s}  {sz/1024:.1f} KB')
print('\n🎉 Pipeline complete!')

---
## 📋 System Summary

| Step | Component | Technology |
|------|-----------|------------|
| 1 | Data Collection | Google Drive PlantVillage (nested) |
| 2 | Data Preprocessing | tf.data + Augmentation + Validation |
| 3 | Disease Detection | MobileNetV2 Transfer Learning + Fine-tuning |
| 4 | Report Generation | CSV Database + Structured Report |
| 5 | Mapping & Visualization | Folium + HeatMap + Cluster Markers |
| 6 | Outbreak Analysis | DBSCAN Hotspot + Time Trend |
| 7 | Alerts & Notifications | SMS/Email Alert Generator |
| 8 | User Dashboard | ipywidgets Interactive App |
| 9 | Feedback & Retraining | Feedback CSV + Accuracy Check |